In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F

catalog_name = 'ecommerce'

## BRANDS

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze.show(5)


In [0]:
df_silver = df_bronze.withColumn('brand_name',F.trim(F.col('brand_name')))

df_silver = df_silver.withColumn('brand_code',F.regexp_replace(F.col('brand_code'),r'[^a-zA-Z0-9]',''))

df_silver.select("category_code").distinct().show()




In [0]:
#anomalies dictinonary

anomalies = {
    "GROCERY": "GRCY",
    "BOOKS":"BKS",
    "TOYS":"TOY"

    }

df_silver = df_silver.replace(anomalies,subset='category_code')
df_silver.select("category_code").distinct().show()    

In [0]:
df_silver.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(f"{catalog_name}.silver.slv_brands")

## CATEGORY

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")
df_bronze.show(10)

In [0]:
%sql
select category_code, count(*) as count from ecommerce.bronze.brz_category group by category_code order by count desc

In [0]:
df_duplicates = df_bronze.groupby("category_code").count().filter("count > 1")
df_duplicates.show()



In [0]:
df_silver = df_bronze.dropDuplicates(["category_code"])
df_silver = df_silver.withColumn('category_name',F.trim(F.col('category_name')))
df_silver = df_silver.withColumn('category_code',F.upper(F.col('category_code')))
display(df_silver)

In [0]:
df_silver.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(f"{catalog_name}.silver.slv_category")

## PRODUCT

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_products")
display(df_bronze.limit(10))


In [0]:
df_duplicates = df_bronze.groupby("product_id").count().filter("count > 1")
df_duplicates.show()

In [0]:
df_silver = df_bronze.withColumn("category_code",F.upper(F.col("category_code")))
df_silver = df_silver.withColumn("brand_code",F.upper(F.col("brand_code")))

## replace g from weight_grams
df_silver = df_silver.withColumn("weight_grams",F.regexp_replace(F.col("weight_grams"), "g", "").cast(IntegerType()))

## replace cm from length_cm
df_silver = df_silver.withColumn("length_cm",F.regexp_replace(F.col("length_cm"), ",", ".").cast(FloatType()))

## fixing spelling
df_silver = df_silver.withColumn("material",
                                 F.when(F.col("material") == "Coton", "Cotton")
                                 .when(F.col("material") == "Alumium", "Aluminium")
                                 .when(F.col("material") == "Ruber", "Rubber")
                                 .otherwise(F.col("material")))


df_silver.show(20)
##display(df_silver.limit(10))


In [0]:
# Convert negative rating_count to positive
df_silver = df_silver.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
     .otherwise(F.lit(0))  # if null, replace with 0
)

## check all changed values
df_silver.select(
    "weight_grams",
    "length_cm",
    "category_code",
    "brand_code",
    "material",
    "rating_count"
).show(10, truncate=False)

In [0]:
df_silver.write.format("delta").mode("overwrite").option("mergeSchema","true").saveAsTable(f"{catalog_name}.silver.slv_products")

## CUSTOMER

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_calendar)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_customers")

# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

df_bronze.show(10)

In [0]:
## Check for null values in cutomer_id coloumn
null_count = df_bronze.filter(F.col("customer_id").isNull()).count()
null_count

# There are 300 null values in customer_id column. Display some of those
df_bronze.filter(F.col("customer_id").isNull()).show(3)

In [0]:
# Drop rows where 'customer_id' is null
df_silver = df_bronze.dropna(subset=["customer_id"])

# Get row count
row_count = df_silver.count()
print(f"Row count after droping null values: {row_count}")

In [0]:
## handle null in phone column

null_count = df_silver.filter(F.col("phone").isNull()).count()
print(f"Number of nulls in phone: {null_count}") 

In [0]:
df_silver.filter(F.col("phone").isNull()).show(3)

In [0]:
### Fill null values with 'Not Available'
df_silver = df_silver.fillna("Not Available", subset=["phone"])

# sanity check (If any nulls still exist)
df_silver.filter(F.col("phone").isNull()).show()

In [0]:
# Write raw data to the silver layer 
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

## DATE

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_calendar)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_date")

# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

df_bronze.show(3)

In [0]:
## converting into date format
from pyspark.sql.functions import to_date


df_silver = df_bronze.withColumn("date", to_date(df_bronze["date"],"dd-MM-yyyy"))

print(df_silver.printSchema())
df_silver.show(3)


In [0]:
# Find duplicate rows in the DataFrame
duplicates = df_silver.groupBy('date').count().filter("count > 1")

# Show the duplicate rows
print("Total duplicated Rows: ", duplicates.count())
display(duplicates)

In [0]:
# Remove duplicate rows
df_silver = df_silver.dropDuplicates(['date'])

# Capitalize first letter of each word in day_name
df_silver = df_silver.withColumn("day_name", F.initcap(F.col("day_name")))

df_silver.show(5)

In [0]:
df_silver = df_silver.withColumn("week_of_year", F.abs(F.col("week_of_year")))  # Convert negative to positive

# enhance `quarter` and `week_of_year` column
df_silver = df_silver.withColumn("quarter", F.concat_ws("", F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))))
df_silver = df_silver.withColumn("week_of_year", F.concat_ws("-", F.concat(F.lit("Week"), F.col("week_of_year"), F.lit("-"), F.col("year"))))

# Rename a column
df_silver = df_silver.withColumnRenamed("week_of_year", "week")

df_silver.show(10)

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_date")